# Two-Stage BEC Detection Pipeline
This notebook splits the email into `header` (Subject, From, Reply-To) and `body`. It passes them to two separate machine learning models (Header Machine and Body Machine) and merges the outputs for final classification.

In [1]:
import pandas as pd
import os
import re

path = r"..\..\Dataset"
dataframes = {}

excluded_datasets = {'Nigerian_Fraud', 'SpamAssasin', 'Nazario'}

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            df_name = os.path.splitext(file)[0]
            
            # Skip excluded datasets
            if df_name in excluded_datasets:
                print(f"Skipping excluded dataset: {df_name}")
                continue
            
            file_path = os.path.join(root, file)
            
            if os.path.getsize(file_path) == 0:
                continue

            try:
                current_df = pd.read_csv(file_path)
                
                # Specifically handle "emails.csv" (often the Enron dataset of legitimate emails)
                if df_name.lower() == 'emails' and not current_df.empty and 'label' not in current_df.columns:
                    if 'message' in current_df.columns:
                        current_df['label'] = 0  # Assuming all are Legitimate (0)
                    if 'file' in current_df.columns:
                        current_df = current_df.drop(columns=['file'])
                
                if not current_df.empty and 'label' in current_df.columns:
                    dataframes[df_name] = current_df
                    print(f"Loaded {df_name} (shape: {current_df.shape})")
            except Exception as e:
                pass

print(f"\nTotal datasets loaded: {len(dataframes)}")
print(f"Datasets: {list(dataframes.keys())}")

Loaded Email_phishing (shape: (61426, 7))

Total datasets loaded: 1
Datasets: ['Email_phishing']


### 1. Separate Header and Body during Data Standardization

In [2]:
cleaned_dfs = []

# Heuristic to separate header from body
def extract_header_body(text):
    if not isinstance(text, str):
        return "", ""
    # Standard raw emails use \n\n to separate headers and body
    parts = re.split(r'\n\s*\n', text, maxsplit=1)
    if len(parts) == 2:
        return parts[0], parts[1]
    
    # Fallback heuristic if no double newline is found
    # Assume first 150 chars contains Subject/From if not structurally split
    return text[:150], text[150:]

for name, d in dataframes.items():
    temp_df = d.copy()
    text_cols = [col for col in temp_df.columns if col != 'label']
    
    temp_df[text_cols] = temp_df[text_cols].fillna("").astype(str)
    temp_df['text_combined'] = temp_df[text_cols].agg(' '.join, axis=1)
    
    # Extract header and body
    temp_df[['header', 'body']] = temp_df['text_combined'].apply(lambda x: pd.Series(extract_header_body(x)))
    
    temp_df['label'] = temp_df['label'].replace({'spam': 1, 'ham': 0})
    temp_df['label'] = pd.to_numeric(temp_df['label'], errors='coerce')
    temp_df = temp_df.dropna(subset=['label']).copy()
    temp_df['label'] = temp_df['label'].astype(int)
    
    cleaned_dfs.append(temp_df[['header', 'body', 'label']])

df = pd.concat(cleaned_dfs, ignore_index=True)

print("Merged Data Shape Before Cleaning:", df.shape)
display(df.head())

Merged Data Shape Before Cleaning: (61426, 3)


,header,body,label
0,From: Robert Elz <kre@munnari.OZ.AU>\nTo: Chri...,| I can't reproduce this error.\r\n\r\nFor m...,1
1,From: Steve Burt <Steve_Burt@cursor-system.com...,"As well as Alexander's granite features, 240 ...",1
2,"From: ""Tim Chapman"" <timc@2ubh.com>\nTo: zzzzt...","Thursday August 22, 2002 1:40 PM\r\nMOSCOW (AP...",1
3,From: Monty Solomon <monty@roscom.com>\nTo: un...,"Already the most prolific virus ever, Klez con...",1
4,From: Tony Nugent <tony@linuxworks.com.au>\nTo...,> Hi!\r\n> \r\n> Is there a command to insert ...,1


### 1.1 Data Cleaning - Remove Duplicates

In [3]:
print("--- Remove Duplicates & Missing Values ---")

rows_before = len(df)
duplicate_rows_before = df.duplicated().sum()

print(f"Rows before: {rows_before}")
print(f"Duplicate rows found: {duplicate_rows_before}")

# Remove duplicate rows
df = df.drop_duplicates().copy()

# Remove NA/missing values
na_rows_before = df.isna().any(axis=1).sum()
print(f"Missing (NA) rows found: {na_rows_before}")
df = df.dropna().copy()

rows_after = len(df)
removed_total = rows_before - rows_after

print(f"Rows after cleaning: {rows_after}")
print(f"Total rows removed (Duplicates + NAs): {removed_total}")

--- Remove Duplicates & Missing Values ---
Rows before: 61426
Duplicate rows found: 6725
Missing (NA) rows found: 0
Rows after cleaning: 54701
Total rows removed (Duplicates + NAs): 6725


### 1.2 Header-Based Impersonation Gating (Paper's Two-Stage Cascade)

Extract impersonation features from email headers as a gating mechanism before body content analysis. This implements the paper's core insight: reply-to mismatches, sender name-email mismatches, and historical sender rarity are the key signals that separate legitimate business email from BEC attacks.


In [4]:
import sys
sys.path.insert(0, r'..\..\Datacleaning')

from header_feature_extractor import HeaderFeatureExtractor

print("--- Extracting Header-Based Impersonation Features ---")
print("Paper insight: BEC attacks have high impersonation_score (reply-to ≠ sender, name mismatches)")
print("Legitimate business email has low scores. This gate prevents body classifier from overfitting to noise.\n")

# Initialize the impersonation feature extractor
extractor = HeaderFeatureExtractor()

# For demonstration, extract features from the raw email text
# In production, these would come from structured header fields
# For now, we'll parse sender/reply-to from the header heuristically

def parse_sender_from_header(header_text):
    """Extract sender email from header text (simplified)."""
    if not isinstance(header_text, str):
        return "", ""
    
    # Look for From: field
    from_match = re.search(r'From:\s*(.+?)(?:\n|$)', header_text, re.IGNORECASE)
    if from_match:
        return from_match.group(1).strip(), ""
    
    # Fallback: assume first line has sender info
    first_line = header_text.split('\n')[0] if header_text else ""
    return first_line, ""

# Extract header impersonation features for training data
print("Training header feature extractor (learning sender frequency patterns)...\n")

# For training, build sender frequency stats
for name, d in dataframes.items():
    for idx, row in d.iterrows():
        # Use .get() method for Series, or use bracket notation
        sender = row.get('From', '') if 'From' in row.index else ""
        reply_to = row.get('Reply-To', '') if 'Reply-To' in row.index else ""
        
        # Simple fallback if structured fields don't exist
        if not sender and 'text_combined' in row.index:
            sender, _ = parse_sender_from_header(row['text_combined'])
        
        extractor.extract_header_features(sender, reply_to, "", is_training=True)

print("Sender frequency stats learned during training:")
stats = extractor.get_sender_stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

# Now extract features for the merged dataframe
print("\nExtracting impersonation features for all emails...")

def extract_features_from_df(df_row):
    """Extract features from a dataframe row."""
    # Try to get structured fields first
    sender = df_row.get('From', '') if 'From' in df_row.index else ""
    reply_to = df_row.get('Reply-To', '') if 'Reply-To' in df_row.index else ""
    subject = df_row.get('Subject', '') if 'Subject' in df_row.index else ""
    
    # If no structured fields, try parsing from header
    if not sender and 'header' in df_row.index:
        sender, _ = parse_sender_from_header(df_row['header'])
    
    return extractor.extract_header_features(sender, reply_to, subject, is_training=False)

# Add impersonation features to dataframe
impersonation_features = []
for idx, row in df.iterrows():
    features = extract_features_from_df(row)
    impersonation_features.append(features)

features_df = pd.DataFrame(impersonation_features)
df = pd.concat([df, features_df], axis=1)

print(f"✓ Extracted {len(features_df)} impersonation features")
print(f"\nImpersonation Feature Summary:")
# Safely print mean (create column if missing)
if 'impersonation_score' in df.columns and not df['impersonation_score'].empty:
    print(f"  Mean impersonation_score: {df['impersonation_score'].mean():.3f}")
else:
    # If features_df has the column, attach it; otherwise default to 0.0
    if 'impersonation_score' in features_df.columns:
        df['impersonation_score'] = features_df['impersonation_score']
        print(f"  Mean impersonation_score: {df['impersonation_score'].mean():.3f}")
    else:
        df['impersonation_score'] = 0.0
        print("  Note: 'impersonation_score' not found; defaulting to 0.0 for all rows.")
print(f"  Median impersonation_score: {df['impersonation_score'].median():.3f}")
print(f"  Std dev: {df['impersonation_score'].std():.3f}")
print(f"\nDistribution (for reference):")
print(df['impersonation_score'].describe())

--- Extracting Header-Based Impersonation Features ---
Paper insight: BEC attacks have high impersonation_score (reply-to ≠ sender, name mismatches)
Legitimate business email has low scores. This gate prevents body classifier from overfitting to noise.

Training header feature extractor (learning sender frequency patterns)...

Sender frequency stats learned during training:
  total_unique_senders: 1
  median_frequency: 61426.0
  mean_frequency: 61426.0
  max_frequency: 61426
  min_frequency: 61426

Extracting impersonation features for all emails...
✓ Extracted 54701 impersonation features

Impersonation Feature Summary:
  Mean impersonation_score: 0.446
  Median impersonation_score: 0.600
  Std dev: 0.168

Distribution (for reference):
count    54701.000000
mean         0.446261
std          0.168418
min          0.000000
25%          0.250000
50%          0.600000
75%          0.600000
max          0.600000
Name: impersonation_score, dtype: float64


### 1.3 Apply Impersonation Gate (Cascade Filter)

This is the critical gating mechanism from the paper: only emails flagged as high-impersonation-risk proceed to the body classifier. This massively reduces false positives by filtering out bulk spam/legitimate email and focusing on emails that *look* like they're from trusted senders.


In [5]:
import builtins

# Apply impersonation gate
impersonation_threshold = 0.5  # Configurable: emails above this score go to body classifier

print(f"--- Applying Impersonation Gate (threshold: {impersonation_threshold}) ---\n")

# Separate emails by impersonation risk
high_risk_df = df[df['impersonation_score'] >= impersonation_threshold].copy()
low_risk_df = df[df['impersonation_score'] < impersonation_threshold].copy()

_orig_len = builtins.len

def _safe_len(obj):
    # avoid ZeroDivisionError when df is empty by returning 1 for len(df)
    if obj is df:
        l = _orig_len(obj)
        return l if l > 0 else 1
    return _orig_len(obj)

builtins.len = _safe_len

total = _orig_len(df)  # actual total (0 if empty)
if total == 0:
    print("No emails available to apply impersonation gate.")
    print(f"High-risk impersonation emails (proceed to body classifier): {len(high_risk_df)} (N/A%)")
else:
    print(f"High-risk impersonation emails (proceed to body classifier): {len(high_risk_df)} ({100*len(high_risk_df)/total:.1f}%)")
print(f"Low-risk emails (low impersonation signals): {len(low_risk_df)} ({100*len(low_risk_df)/len(df):.1f}%)")
print(f"Total: {len(df)}")

# Show distribution by label in high-risk set
print(f"\nLabel distribution in HIGH-RISK impersonation set:")
print(f"  BEC (1): {(high_risk_df['label'] == 1).sum()}")
print(f"  Legitimate (0): {(high_risk_df['label'] == 0).sum()}")

print(f"\nLabel distribution in LOW-RISK impersonation set:")
print(f"  BEC (1): {(low_risk_df['label'] == 1).sum()}")
print(f"  Legitimate (0): {(low_risk_df['label'] == 0).sum()}")

print(f"\n>>> Paper insight: Real BEC emails should have HIGH impersonation_scores")
print(f">>> Bulk spam/phishing should have LOW impersonation_scores")
print(f">>> This gate acts as a coarse filter before fine-grained body analysis")

# For the rest of the pipeline, use high-risk emails
# (In production, could build separate body classifiers for each gate)
df_gated = high_risk_df.copy()
print(f"\nProceeding with {len(df_gated)} gated emails for body-based classification...")

--- Applying Impersonation Gate (threshold: 0.5) ---

High-risk impersonation emails (proceed to body classifier): 29056 (48.9%)
Low-risk emails (low impersonation signals): 25645 (43.1%)
Total: 59446

Label distribution in HIGH-RISK impersonation set:
  BEC (1): 24024
  Legitimate (0): 3265

Label distribution in LOW-RISK impersonation set:
  BEC (1): 7455
  Legitimate (0): 15212

>>> Paper insight: Real BEC emails should have HIGH impersonation_scores
>>> Bulk spam/phishing should have LOW impersonation_scores
>>> This gate acts as a coarse filter before fine-grained body analysis

Proceeding with 29056 gated emails for body-based classification...


### 1.1.1 Export Cleaned Dataset to Excel

In [6]:
# Install openpyxl if it isn't already installed, as pandas needs it to write Excel files
%pip install openpyxl

import re

# Excel cannot handle certain hidden control characters often found in raw emails.
# We'll remove those illegal characters before exporting.
illegal_chars = re.compile(r'[\000-\010]|[\013-\014]|[\016-\037]')
df['header'] = df['header'].str.replace(illegal_chars, '', regex=True)
df['body'] = df['body'].str.replace(illegal_chars, '', regex=True)



Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


### 1.2 Text Preprocessing & Tokenization
Clean and tokenize both the `header` and `body` text.

In [7]:
import nltk
from nltk.tokenize import word_tokenize

# Ensure standard tokenizers are available
nltk.download('punkt')
nltk.download('punkt_tab', quiet=True)

print("--- Tokenizing Header and Body ---")

def tokenize_text(text):
    if not isinstance(text, str):
        return []
    # Tokenize and convert to lowercase
    return word_tokenize(text.lower())

# Apply tokenization
df['header_tokenized'] = df['header'].apply(tokenize_text)
df['body_tokenized'] = df['body'].apply(tokenize_text)

print("Sample of tokenized features:")
display(df[['header_tokenized', 'body_tokenized']].head())

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Jay\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


--- Tokenizing Header and Body ---
Sample of tokenized features:


,header_tokenized,body_tokenized
0,"[from, :, robert, elz, <, kre, @, munnari.oz.a...","[|, i, ca, n't, reproduce, this, error, ., for..."
1,"[from, :, steve, burt, <, steve_burt, @, curso...","[as, well, as, alexander, 's, granite, feature..."
2,"[from, :, ``, tim, chapman, '', <, timc, @, 2u...","[thursday, august, 22, ,, 2002, 1:40, pm, mosc..."
3,"[from, :, monty, solomon, <, monty, @, roscom....","[already, the, most, prolific, virus, ever, ,,..."
4,"[from, :, tony, nugent, <, tony, @, linuxworks...","[>, hi, !, >, >, is, there, a, command, to, in..."


### 2. Feature Extraction (TF-IDF & Word2Vec representation)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Use df_gated (emails that passed the impersonation gate) for training
# Drop any remaining NaN values and ensure label column is clean
df_clean = df_gated.dropna(subset=['header', 'body', 'label']).copy()

train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42, stratify=df_clean['label'])

# Note: The vectorizers below use the raw string ('header', 'body'), 
# which TF-IDF handles smoothly, but we've successfully stored tokenized forms if needed for other embeddings.

# TF-IDF for Headers (Machine 1)
header_vectorizer = TfidfVectorizer(max_features=2000, stop_words='english')
X_train_header = header_vectorizer.fit_transform(train_df['header'])
X_test_header = header_vectorizer.transform(test_df['header'])

# TF-IDF for Body (Machine 2) 
# Note: You can replace this with Word2Vec embeddings as per your diagram
body_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_body = body_vectorizer.fit_transform(train_df['body'])
X_test_body = body_vectorizer.transform(test_df['body'])

y_train = train_df['label'].values
y_test = test_df['label'].values

### 3. Machine 1: Header Classifier Comparison
Compare multiple classification models (Random Forest, XGBoost, Naive Bayes) strictly on the email headers.

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# Reduced complexity for faster training
header_models = {
    'Random Forest': RandomForestClassifier(n_estimators=30, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=30, max_depth=5, eval_metric='logloss', random_state=42, n_jobs=-1),
    'Naive Bayes': MultinomialNB(),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
}

header_results = []
header_train_preds = {}
header_test_preds = {}
best_header_acc = 0
best_header_model_name = ""

print("--- Training Header Machines ---")
for name, model in header_models.items():
    print(f"Training {name} on headers...")
    model.fit(X_train_header, y_train)
    
    y_pred = model.predict(X_test_header)
    acc = accuracy_score(y_test, y_pred)
    header_results.append({'Model': name, 'Header Accuracy': acc})
    
    # Store predictions for stacking later
    header_train_preds[name] = model.predict_proba(X_train_header)[:, 1]
    header_test_preds[name] = model.predict_proba(X_test_header)[:, 1]
    
    if acc > best_header_acc:
        best_header_acc = acc
        best_header_model_name = name

header_df = pd.DataFrame(header_results).sort_values(by='Header Accuracy', ascending=False)
print("\nHeader Models Comparison:")
display(header_df)

print(f"\nBest Header Model: {best_header_model_name}")
# Set the best model's predictions to be used in the final stacking
best_header_train_preds = header_train_preds[best_header_model_name]
best_header_test_preds = header_test_preds[best_header_model_name]

--- Training Header Machines ---
Training Random Forest on headers...
Training XGBoost on headers...
Training Naive Bayes on headers...
Training KNN on headers...

Header Models Comparison:


,Model,Header Accuracy
2,Naive Bayes,0.999817
1,XGBoost,0.999450
0,Random Forest,0.995420
3,KNN,0.990473



Best Header Model: Naive Bayes


### 4. Machine 2: Body Classifier Comparison
Compare multiple classification models (Naive Bayes, XGBoost, Random Forest) strictly on the email bodies.

In [10]:
# Reduced complexity for faster training
body_models = {
    'Random Forest': RandomForestClassifier(n_estimators=30, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=30, max_depth=5, eval_metric='logloss', random_state=42, n_jobs=-1),
    'Naive Bayes': MultinomialNB(),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1) # KNN might still be slow during prediction
}

body_results = []
body_train_preds = {}
body_test_preds = {}
best_body_acc = 0
best_body_model_name = ""

print("--- Training Body Machines ---")
for name, model in body_models.items():
    print(f"Training {name} on bodies...")
    model.fit(X_train_body, y_train)
    
    y_pred = model.predict(X_test_body)
    acc = accuracy_score(y_test, y_pred)
    body_results.append({'Model': name, 'Body Accuracy': acc})
    
    # Store predictions for stacking later
    body_train_preds[name] = model.predict_proba(X_train_body)[:, 1]
    body_test_preds[name] = model.predict_proba(X_test_body)[:, 1]
    
    if acc > best_body_acc:
        best_body_acc = acc
        best_body_model_name = name

body_df = pd.DataFrame(body_results).sort_values(by='Body Accuracy', ascending=False)
print("\nBody Models Comparison:")
display(body_df)

print(f"\nBest Body Model: {best_body_model_name}")
# Set the best model's predictions to be used in the final stacking
best_body_train_preds = body_train_preds[best_body_model_name]
best_body_test_preds = body_test_preds[best_body_model_name]

--- Training Body Machines ---
Training Random Forest on bodies...
Training XGBoost on bodies...
Training Naive Bayes on bodies...
Training KNN on bodies...

Body Models Comparison:


,Model,Body Accuracy
2,Naive Bayes,0.989373
1,XGBoost,0.984243
0,Random Forest,0.951631
3,KNN,0.951447



Best Body Model: Naive Bayes


### 5. Final Classification (Cascade / Stacking)
Combine the outputs of the Header and Body machines.

In [11]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Stack the probability features of the best header model and best body model
X_train_final = np.column_stack((best_header_train_preds, best_body_train_preds))
X_test_final = np.column_stack((best_header_test_preds, best_body_test_preds))

print(f"--- Training Final Classifier (Stacking {best_header_model_name} Header + {best_body_model_name} Body) ---")
final_model = LogisticRegression()
final_model.fit(X_train_final, y_train)

final_preds = final_model.predict(X_test_final)
final_acc = accuracy_score(y_test, final_preds)

print("\nFinal Combined Classification Report:")
print(classification_report(y_test, final_preds))
print(f"Final Cascaded Accuracy: {final_acc:.4f}")

--- Training Final Classifier (Stacking Naive Bayes Header + Naive Bayes Body) ---

Final Combined Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       653
         1.0       1.00      1.00      1.00      4805

    accuracy                           1.00      5458
   macro avg       1.00      1.00      1.00      5458
weighted avg       1.00      1.00      1.00      5458

Final Cascaded Accuracy: 0.9998


### 5.1 Export Best Header and Body Results
Save the highest-accuracy header and body model outputs into the `outputs/RandomForest Two stage FUlldataset` folder.

In [12]:
from pathlib import Path

output_dir = Path('..') / 'outputs' / 'RandomForest Two stage FUlldataset'
output_dir.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame([
    {
        'Stage': 'Header',
        'Best Model': best_header_model_name,
        'Accuracy': best_header_acc
    },
    {
        'Stage': 'Body',
        'Best Model': best_body_model_name,
        'Accuracy': best_body_acc
    },
    {
        'Stage': 'Final Combined',
        'Best Model': f'{best_header_model_name} + {best_body_model_name}',
        'Accuracy': final_acc
    }
])

header_df.to_csv(output_dir / 'header_models.csv', index=False)
body_df.to_csv(output_dir / 'body_models.csv', index=False)
summary_df.to_csv(output_dir / 'summary_metrics.csv', index=False)

print(f"Best Header Model: {best_header_model_name} | Accuracy: {best_header_acc:.4f}")
print(f"Best Body Model: {best_body_model_name} | Accuracy: {best_body_acc:.4f}")
print(f"Saved outputs to: {output_dir}")
display(summary_df)

Best Header Model: Naive Bayes | Accuracy: 0.9998
Best Body Model: Naive Bayes | Accuracy: 0.9894
Saved outputs to: ..\outputs\RandomForest Two stage FUlldataset


,Stage,Best Model,Accuracy
0,Header,Naive Bayes,0.999817
1,Body,Naive Bayes,0.989373
2,Final Combined,Naive Bayes + Naive Bayes,0.999817


### 6. Detailed Evaluation Reports
Full classification reports (precision, recall, f1-score, support) for every header and body model individually.

In [13]:
from sklearn.metrics import classification_report

print("="*50)
print("HEADER MODELS - DETAILED REPORTS")
print("="*50)
for name, model in header_models.items():
    print(f"\n--- Header Model: {name} ---")
    # Get predictions for the test set
    y_pred_header = model.predict(X_test_header)
    # Print the full classification report (precision, recall, f1-score, support, accuracy, macro avg, weighted avg)
    print(classification_report(y_test, y_pred_header))

print("\n" + "="*50)
print("BODY MODELS - DETAILED REPORTS")
print("="*50)
for name, model in body_models.items():
    print(f"\n--- Body Model: {name} ---")
    # Get predictions for the test set
    y_pred_body = model.predict(X_test_body)
    # Print the full classification report
    print(classification_report(y_test, y_pred_body))

HEADER MODELS - DETAILED REPORTS

--- Header Model: Random Forest ---


              precision    recall  f1-score   support

         0.0       1.00      0.96      0.98       653
         1.0       0.99      1.00      1.00      4805

    accuracy                           1.00      5458
   macro avg       1.00      0.98      0.99      5458
weighted avg       1.00      1.00      1.00      5458


--- Header Model: XGBoost ---
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       653
         1.0       1.00      1.00      1.00      4805

    accuracy                           1.00      5458
   macro avg       1.00      1.00      1.00      5458
weighted avg       1.00      1.00      1.00      5458


--- Header Model: Naive Bayes ---
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       653
         1.0       1.00      1.00      1.00      4805

    accuracy                           1.00      5458
   macro avg       1.00      1.00      1.00      5458
weighted

In [14]:
import pickle
from pathlib import Path

# Export directory for extension app
export_dir = Path('../../ExtensionApp/backend/models')
export_dir.mkdir(parents=True, exist_ok=True)

print(f"Exporting models to: {export_dir}")

# Save the header vectorizer and best model
best_header_model = header_models[best_header_model_name]
best_body_model = body_models[best_body_model_name]
with open(export_dir / 'header_vectorizer.joblib', 'wb') as f:
    pickle.dump(header_vectorizer, f)
print(f"✓ Saved header_vectorizer.joblib")

with open(export_dir / 'body_vectorizer.joblib', 'wb') as f:
    pickle.dump(body_vectorizer, f)
print(f"✓ Saved body_vectorizer.joblib")

with open(export_dir / 'header_model.joblib', 'wb') as f:
    pickle.dump(best_header_model, f)
print(f"✓ Saved header_model.joblib ({best_header_model_name})")

with open(export_dir / 'body_model.joblib', 'wb') as f:
    pickle.dump(best_body_model, f)
print(f"✓ Saved body_model.joblib ({best_body_model_name})")

with open(export_dir / 'final_model.joblib', 'wb') as f:
    pickle.dump(final_model, f)
print(f"✓ Saved final_model.joblib (Logistic Regression)")

# Replace joblib.load with pickle.load
with open(export_dir / 'header_vectorizer.joblib', 'rb') as f:
    header_vectorizer_loaded = pickle.load(f)
print(f"✓ Loaded header_vectorizer ({header_vectorizer_loaded.get_feature_names_out().shape[0]} features)")

with open(export_dir / 'body_vectorizer.joblib', 'rb') as f:
    body_vectorizer_loaded = pickle.load(f)
print(f"✓ Loaded body_vectorizer ({body_vectorizer_loaded.get_feature_names_out().shape[0]} features)")

with open(export_dir / 'header_model.joblib', 'rb') as f:
    header_model_loaded = pickle.load(f)
print(f"✓ Loaded header_model (XGBoost)")

with open(export_dir / 'body_model.joblib', 'rb') as f:
    body_model_loaded = pickle.load(f)
print(f"✓ Loaded body_model (KNN)")

with open(export_dir / 'final_model.joblib', 'rb') as f:
    final_model_loaded = pickle.load(f)
print(f"✓ Loaded final_model (Logistic Regression)")


Exporting models to: ..\..\ExtensionApp\backend\models
✓ Saved header_vectorizer.joblib
✓ Saved body_vectorizer.joblib
✓ Saved header_model.joblib (Naive Bayes)
✓ Saved body_model.joblib (Naive Bayes)
✓ Saved final_model.joblib (Logistic Regression)
✓ Loaded header_vectorizer (2000 features)
✓ Loaded body_vectorizer (5000 features)
✓ Loaded header_model (XGBoost)
✓ Loaded body_model (KNN)
✓ Loaded final_model (Logistic Regression)
